# Glucast — Reentrenamiento y comparativa de modelos
## TFM — Máster Universitario en Inteligencia Artificial (UNIR)
**Autor:** Pablo Daniel Figueroa

Este notebook arma el **capítulo de resultados de la Entrega 2**. Con los datos actualizados de Nightscout compara, sobre **el mismo conjunto de test** (el último 15 % cronológico de los datos propios):

| # | Modelo | Qué responde |
|---|---|---|
| 1 | **Persistencia** (la glucosa no cambia) | Línea base mínima que cualquier modelo debe superar |
| 2 | **ARIMA(p,1,0)** | Línea base estadística clásica |
| 3 | **GRU** solo con datos propios | ¿Importa la celda recurrente (GRU vs LSTM)? |
| 4 | **LSTM** solo con datos propios | Modelo personal sin datos externos |
| 5 | **LSTM dataset unificado** (OhioT1DM + propios en un único dataset) | Sugerencia del profesor de Redes Neuronales |
| 6 | **LSTM transfer learning** (pre-entrena OhioT1DM → *fine-tuning* propio) | Estrategia original del TFM |

Además calcula:
- **RMSE, MAE, MARD** y **zonas de Clarke** (A–E), con **intervalos de confianza al 95 %** por *bootstrap* por días.
- **Incertidumbre de la predicción** con *Monte Carlo Dropout* (banda al 95 % y su cobertura real).
- Horizontes de **30 y 60 minutos**.

**Decisiones de diseño para que la comparación sea justa**
- Todos los modelos usan **las mismas variables**, presentes en ambas fuentes: glucosa, tendencia, bolos, IOB, carbohidratos, COB y hora del día. Se excluyen FC/pasos/Garmin porque no significan lo mismo en las dos fuentes (Ohio: FC cada 5 min; Garmin: resumen diario).
- **Escalado fijo con unidades fisiológicas** (por ejemplo glucosa/400), igual para todos los pacientes. Así no hay fuga de información del test y las dos fuentes quedan en la misma escala.
- Las redes predicen el **cambio de glucosa** (Δ) respecto del último valor; después se reconstruye la glucosa absoluta.
- Solo se usan ventanas **sin huecos de sensor** (se interpolan como máximo 15 min).

> **Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*. Con GPU todo tarda ~15–30 min.

## 1. Librerías

In [ ]:
import os, glob, json, time, zipfile, getpass, pickle, warnings
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
warnings.filterwarnings('ignore')

SEMILLA = 42
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)
plt.rcParams['figure.figsize'] = (13, 4.5)
plt.rcParams['font.size'] = 11
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU') or 'NO (va a tardar más)')

## 2. Configuración

In [ ]:
NIGHTSCOUT_URL = 'https://TU-NIGHTSCOUT.up.railway.app'
FECHA_DESDE    = None          # p. ej. '2026-01-01' para limitar; None = todo lo que haya
ZONA_HORARIA   = 'America/Argentina/Buenos_Aires'

VENTANA    = 24                # 2 h de historia (24 x 5 min)
HORIZONTES = [6, 12]           # 30 y 60 min. Para ir más rápido: [6]
SPLIT      = (0.70, 0.15, 0.15)  # train / validación / test (cronológico)

EPOCHS       = 100
LR_INICIAL   = 1e-3
LR_FINETUNE  = 1e-4
PESO_PROPIO_UNIFICADO = 'balanceado'  # en el dataset unificado, los datos propios pesan lo mismo que todo Ohio junto. Usar 1.0 para peso igual por muestra.
MC_MUESTRAS  = 50              # pasadas de Monte Carlo Dropout
BOOTSTRAP_B  = 1000

CARPETA_DRIVE = '/content/drive/MyDrive/MAESTRIA EN IA/TFM/ENTRENAMIENTO'
CARPETA_RES   = os.path.join(CARPETA_DRIVE, 'resultados_' + datetime.now().strftime('%Y%m%d_%H%M'))

FEATURES = ['glucosa', 'tendencia', 'bolo_u', 'iob', 'carbs', 'cob', 'hora_sin', 'hora_cos']
ESCALAS  = {'glucosa': 400, 'tendencia': 50, 'bolo_u': 10, 'iob': 10,
            'carbs': 100, 'cob': 100, 'hora_sin': 1, 'hora_cos': 1}
ESCALA_DELTA = 100             # el objetivo (Δ glucosa) se escala /100
DIA_PERIODOS = 24              # DIA 2 h (Fiasp)
ABSORCION_CARBS = 36           # 3 h para COB

## 3. Google Drive y token de Nightscout
Se pide el **token de lectura** (`glucast-app`, el mismo de la app). No hace falta el API secret.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(CARPETA_RES, exist_ok=True)
print('Resultados en:', CARPETA_RES)
NIGHTSCOUT_TOKEN = getpass.getpass('Token de Nightscout (glucast-app): ').strip()

## 4. Descargar los datos propios desde Nightscout

In [ ]:
def ns_get(ruta, params):
    params = dict(params, token=NIGHTSCOUT_TOKEN)
    for intento in range(4):
        try:
            r = requests.get(NIGHTSCOUT_URL.rstrip('/') + ruta, params=params, timeout=120)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if intento == 3:
                raise
            print('  reintento por', e); time.sleep(3)

def descargar_nightscout(dias_bloque=30, max_vacios=3):
    hasta = datetime.now(timezone.utc) + timedelta(days=1)
    limite = pd.Timestamp(FECHA_DESDE, tz='UTC') if FECHA_DESDE else None
    sgv, trt, vacios = [], [], 0
    while vacios < max_vacios:
        desde = hasta - timedelta(days=dias_bloque)
        if limite is not None and hasta <= limite:
            break
        e = ns_get('/api/v1/entries/sgv.json', {'find[date][$gte]': int(desde.timestamp()*1000),
                                                'find[date][$lt]': int(hasta.timestamp()*1000), 'count': 100000})
        t = ns_get('/api/v1/treatments.json', {'find[created_at][$gte]': desde.strftime('%Y-%m-%dT%H:%M:%S.000Z'),
                                               'find[created_at][$lt]': hasta.strftime('%Y-%m-%dT%H:%M:%S.000Z'),
                                               'count': 100000})
        print(f"  {desde:%Y-%m-%d} → {hasta:%Y-%m-%d}: {len(e)} lecturas, {len(t)} tratamientos")
        vacios = vacios + 1 if not e and not t else 0
        sgv += e; trt += t
        hasta = desde
    df_sgv = pd.DataFrame([{'ts': x['date'], 'glucosa': x['sgv']} for x in sgv if x.get('sgv') and x.get('date')])
    df_sgv['ts'] = pd.to_datetime(df_sgv['ts'], unit='ms', utc=True)
    filas = []
    for x in trt:
        if not x.get('created_at'):
            continue
        filas.append({'ts': x['created_at'], 'bolo_u': float(x.get('insulin') or 0), 'carbs': float(x.get('carbs') or 0)})
    df_trt = pd.DataFrame(filas)
    if len(df_trt):
        df_trt['ts'] = pd.to_datetime(df_trt['ts'], utc=True, format='ISO8601')
    return df_sgv, df_trt

print('Descargando de Nightscout...')
df_sgv, df_trt = descargar_nightscout()
df_sgv.to_csv(os.path.join(CARPETA_RES, 'nightscout_sgv_crudo.csv'), index=False)
df_trt.to_csv(os.path.join(CARPETA_RES, 'nightscout_tratamientos_crudo.csv'), index=False)
print(f'\nTotal: {len(df_sgv)} lecturas | {len(df_trt)} tratamientos')

## 5. Preprocesamiento común (mismo código para las dos fuentes)

In [ ]:
def construir_base(gluc, bolos, carbs):
    # gluc/bolos/carbs: Series indexadas por timestamp local (naive)
    g = gluc.groupby(level=0).mean().resample('5min').mean()
    g = g.interpolate(method='linear', limit=3, limit_area='inside')   # huecos de hasta 15 min
    df = pd.DataFrame({'glucosa': g})
    df['bolo_u'] = bolos.resample('5min').sum().reindex(df.index).fillna(0) if len(bolos) else 0.0
    df['carbs']  = carbs.resample('5min').sum().reindex(df.index).fillna(0) if len(carbs) else 0.0
    pesos_iob = np.maximum(0, 1 - np.arange(DIA_PERIODOS) / DIA_PERIODOS)
    pesos_cob = np.maximum(0, 1 - np.arange(ABSORCION_CARBS) / ABSORCION_CARBS)
    df['iob'] = np.convolve(df['bolo_u'].values, pesos_iob)[:len(df)]
    df['cob'] = np.convolve(df['carbs'].values, pesos_cob)[:len(df)]
    df['tendencia'] = df['glucosa'].diff()
    hora = df.index.hour + df.index.minute / 60
    df['hora_sin'] = np.sin(2 * np.pi * hora / 24)
    df['hora_cos'] = np.cos(2 * np.pi * hora / 24)
    return df

def a_local(ts):
    return ts.dt.tz_convert(ZONA_HORARIA).dt.tz_localize(None)

s_g = pd.Series(df_sgv['glucosa'].astype(float).values, index=a_local(df_sgv['ts'])).sort_index()
s_g = s_g[(s_g >= 39) & (s_g <= 401)]
if len(df_trt):
    tl = df_trt.assign(ts=a_local(df_trt['ts'])).set_index('ts').sort_index()
    s_b, s_c = tl['bolo_u'][tl['bolo_u'] > 0], tl['carbs'][tl['carbs'] > 0]
else:
    s_b = s_c = pd.Series(dtype=float)

df_propio = construir_base(s_g, s_b, s_c)
df_propio['fuente'], df_propio['paciente'] = 'propio', 'glucast'
validas = df_propio['glucosa'].notna()
print(f"Datos propios: {df_propio.index.min():%d/%m/%Y} → {df_propio.index.max():%d/%m/%Y}")
print(f"  {validas.sum()} intervalos de 5 min con glucosa ({validas.mean()*100:.1f}% de cobertura)")
print(f"  Bolos: {len(s_b)} | Carbohidratos: {len(s_c)}")
print(f"  Glucosa media {df_propio['glucosa'].mean():.0f} mg/dL | TIR 70-180: {((df_propio['glucosa']>=70)&(df_propio['glucosa']<=180))[validas].mean()*100:.1f}%")

## 6. Cargar OhioT1DM
Si `OhioT1DM.zip` está en la carpeta ENTRENAMIENTO de Drive se usa de ahí; si no, se pide subirlo.

In [ ]:
zip_drive = os.path.join(CARPETA_DRIVE, 'OhioT1DM.zip')
if os.path.exists(zip_drive):
    ruta_zip = zip_drive
else:
    from google.colab import files
    print('Subí OhioT1DM.zip'); files.upload(); ruta_zip = 'OhioT1DM.zip'
PASSWORD = getpass.getpass('Contraseña del dataset OhioT1DM: ').encode()
with zipfile.ZipFile(ruta_zip) as z:
    z.extractall('/content/OhioT1DM', pwd=PASSWORD)
ARCHIVOS_OHIO_TRAIN = sorted(glob.glob('/content/OhioT1DM/**/train/*.xml', recursive=True))
ARCHIVOS_OHIO_TEST  = sorted(glob.glob('/content/OhioT1DM/**/test/*.xml', recursive=True))
print(len(ARCHIVOS_OHIO_TRAIN), 'archivos train |', len(ARCHIVOS_OHIO_TEST), 'archivos test')

In [ ]:
def leer_ohio(ruta):
    raiz = ET.parse(ruta).getroot()
    fmt = '%d-%m-%Y %H:%M:%S'
    def serie(tag, campo_ts, campo_val):
        el = raiz.find(tag)
        if el is None:
            return pd.Series(dtype=float)
        filas = [(ev.get(campo_ts), ev.get(campo_val)) for ev in el.findall('event') if ev.get(campo_val)]
        if not filas:
            return pd.Series(dtype=float)
        idx = pd.to_datetime([f[0] for f in filas], format=fmt)
        return pd.Series([float(f[1]) for f in filas], index=idx).sort_index()
    df = construir_base(serie('glucose_level', 'ts', 'value'),
                        serie('bolus', 'ts_begin', 'dose'),
                        serie('meal', 'ts', 'carbs'))
    df['fuente'], df['paciente'] = 'ohio', 'ohio_' + raiz.get('id')
    return df

ohio_train = [leer_ohio(f) for f in ARCHIVOS_OHIO_TRAIN]
ohio_test  = [leer_ohio(f) for f in ARCHIVOS_OHIO_TEST]
for d in ohio_train:
    print(f"  {d['paciente'].iloc[0]}: {d['glucosa'].notna().sum()} lecturas")

## 7. Dataset unificado y ventanas
Se guarda el dataset unificado (una fila cada 5 min, columna `fuente` y `paciente`) en Drive.

In [ ]:
dataset_unificado = pd.concat([df_propio] + ohio_train + ohio_test)
dataset_unificado.rename_axis('timestamp').to_csv(os.path.join(CARPETA_RES, 'glucast_dataset_unificado.csv'))
print(dataset_unificado.groupby('fuente')['glucosa'].agg(['count', 'mean', 'std']).round(1))

def ventanas(df, H):
    # Devuelve X (n, VENTANA, F) escalado, y (Δ escalado), glucosa actual, glucosa real futura y timestamp del objetivo.
    g = df['glucosa'].values
    F = np.stack([(df[c].fillna(0).values / ESCALAS[c]) for c in FEATURES], axis=1).astype('float32')
    n = len(df) - VENTANA - H + 1
    if n <= 0:
        return None
    valido_g = ~np.isnan(g)
    # ventana completa: VENTANA puntos + el anterior (tendencia) + objetivo
    acum = np.concatenate([[0], np.cumsum(~valido_g)])
    ini = np.arange(n)
    huecos = acum[ini + VENTANA] - acum[np.maximum(ini - 1, 0)]
    ok = (huecos == 0) & (ini >= 1) & valido_g[ini + VENTANA - 1 + H]
    ini = ini[ok]
    X = np.lib.stride_tricks.sliding_window_view(F, (VENTANA, F.shape[1]))[:, 0][ini]
    g_act = g[ini + VENTANA - 1]
    g_fut = g[ini + VENTANA - 1 + H]
    y = ((g_fut - g_act) / ESCALA_DELTA).astype('float32')
    t = df.index[ini + VENTANA - 1 + H]
    return X.astype('float32'), y, g_act, g_fut, t

def juntar(lista, H):
    partes = [v for v in (ventanas(d, H) for d in lista) if v is not None]
    return tuple(np.concatenate([p[i] for p in partes]) for i in range(4)) + (np.concatenate([np.asarray(p[4]) for p in partes]),)

def dividir_propio(H):
    X, y, ga, gf, t = ventanas(df_propio, H)
    t = pd.DatetimeIndex(t)
    t1 = df_propio.index[int(len(df_propio) * SPLIT[0])]
    t2 = df_propio.index[int(len(df_propio) * (SPLIT[0] + SPLIT[1]))]
    tr, va, te = t < t1, (t >= t1 + pd.Timedelta(minutes=5*(VENTANA+H))) & (t < t2), t >= t2 + pd.Timedelta(minutes=5*(VENTANA+H))
    return {k: (X[m], y[m], ga[m], gf[m], t[m]) for k, m in (('train', tr), ('val', va), ('test', te))}

_d = dividir_propio(HORIZONTES[0])
for k, v in _d.items():
    print(f"Propio {k:5s}: {len(v[0]):6d} ventanas  ({v[4].min():%d/%m/%Y} → {v[4].max():%d/%m/%Y})")

## 8. Líneas base: persistencia y ARIMA(p,1,0)
El ARIMA se estima por mínimos cuadrados condicionales sobre las diferencias de glucosa del *train* propio; el orden *p* se elige por BIC (1–8). La predicción a *h* pasos se hace de forma recursiva desde cada ventana.

In [ ]:
def ajustar_arima(X_train, p_max=8):
    gl = X_train[:, :, 0] * ESCALAS['glucosa']
    d = np.diff(gl, axis=1)                       # diferencias dentro de cada ventana (sin cruzar huecos)
    mejor = None
    for p in range(1, p_max + 1):
        Y = d[:, p:].reshape(-1)
        Z = np.stack([d[:, p - i - 1: d.shape[1] - i - 1].reshape(-1) for i in range(p)], axis=1)
        coef, *_ = np.linalg.lstsq(Z, Y, rcond=None)
        res = Y - Z @ coef
        bic = len(Y) * np.log(np.mean(res ** 2)) + p * np.log(len(Y))
        if mejor is None or bic < mejor[0]:
            mejor = (bic, p, coef)
    return mejor[1], mejor[2]

def predecir_arima(X, p, coef, H):
    gl = X[:, :, 0] * ESCALAS['glucosa']
    d = list(np.diff(gl, axis=1)[:, -p:].T)       # últimas p diferencias
    nivel = gl[:, -1].copy()
    for _ in range(H):
        nueva = sum(coef[i] * d[-1 - i] for i in range(p))
        d.append(nueva); nivel = nivel + nueva
    return nivel

## 9. Modelos recurrentes
Misma arquitectura que el modelo original del TFM (LSTM 64 → LSTM 32 → Dense 16 → 1), cambiando solo la celda en la GRU.

In [ ]:
def construir_rnn(tipo='lstm'):
    Celda = LSTM if tipo == 'lstm' else GRU
    m = Sequential([
        Input((VENTANA, len(FEATURES))),
        Celda(64, return_sequences=True), BatchNormalization(), Dropout(0.2),
        Celda(32), BatchNormalization(), Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1),
    ])
    m.compile(optimizer=Adam(LR_INICIAL), loss='mse', metrics=['mae'])
    return m

def entrenar(m, Xtr, ytr, Xva, yva, lr=None, pesos=None, batch=128, nombre=''):
    if lr is not None:
        m.compile(optimizer=Adam(lr), loss='mse', metrics=['mae'])
    t0 = time.time()
    h = m.fit(Xtr, ytr, sample_weight=pesos, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=batch, verbose=0,
              callbacks=[EarlyStopping(patience=12, restore_best_weights=True),
                         ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-6)])
    print(f"  {nombre:28s} {len(h.history['loss']):3d} épocas | val_loss {min(h.history['val_loss']):.5f} | {time.time()-t0:5.0f}s")
    return h

def predecir_glucosa(m, X, g_act):
    return g_act + m.predict(X, batch_size=2048, verbose=0).ravel() * ESCALA_DELTA

## 10. Métricas: RMSE, MAE, MARD, Clarke y bootstrap por días

In [ ]:
def zonas_clarke(ref, pred):
    r, p = np.asarray(ref, float), np.asarray(pred, float)
    A = ((r <= 70) & (p <= 70)) | ((p >= 0.8 * r) & (p <= 1.2 * r))
    E = ((r >= 180) & (p <= 70)) | ((r <= 70) & (p >= 180))
    C = ((r >= 70) & (r <= 290) & (p >= r + 110)) | ((r >= 130) & (r <= 180) & (p <= 1.4 * r - 182))
    D = ((r >= 240) & (p >= 70) & (p <= 180)) | ((r <= 175 / 3) & (p >= 70) & (p <= 180)) | \
        ((r >= 175 / 3) & (r <= 70) & (p >= 1.2 * r))
    z = np.full(r.shape, 'B', dtype='<U1')
    z[D] = 'D'; z[C] = 'C'; z[E] = 'E'; z[A] = 'A'
    return z

def metricas(ref, pred, t, B=BOOTSTRAP_B):
    err = pred - ref
    z = zonas_clarke(ref, pred)
    dias = pd.DatetimeIndex(t).normalize()
    cods, inv = np.unique(dias, return_inverse=True)
    se = np.bincount(inv, err ** 2); ae = np.bincount(inv, np.abs(err)); n = np.bincount(inv)
    rng = np.random.default_rng(SEMILLA)
    idx = rng.integers(0, len(cods), (B, len(cods)))
    rmse_b = np.sqrt(se[idx].sum(1) / n[idx].sum(1))
    mae_b = ae[idx].sum(1) / n[idx].sum(1)
    return {
        'RMSE': np.sqrt(np.mean(err ** 2)), 'RMSE_IC95': f"[{np.percentile(rmse_b, 2.5):.1f}, {np.percentile(rmse_b, 97.5):.1f}]",
        'MAE': np.mean(np.abs(err)), 'MAE_IC95': f"[{np.percentile(mae_b, 2.5):.1f}, {np.percentile(mae_b, 97.5):.1f}]",
        'MARD_%': np.mean(np.abs(err) / ref) * 100,
        'Clarke_A_%': np.mean(z == 'A') * 100, 'Clarke_A+B_%': np.mean(np.isin(z, ['A', 'B'])) * 100,
        'Clarke_C/D/E_%': np.mean(np.isin(z, ['C', 'D', 'E'])) * 100, 'n_test': len(ref),
    }

## 11. Experimento completo (para cada horizonte)
Esta es la celda larga. Muestra el progreso de cada modelo.

In [ ]:
resultados, predicciones, modelos = [], {}, {}

for H in HORIZONTES:
    print(f"\n{'='*70}\nHORIZONTE {H*5} min\n{'='*70}")
    P = dividir_propio(H)
    Xtr, ytr, gatr, _, _ = P['train']; Xva, yva, _, _, _ = P['val']; Xte, yte, gate, gfte, tte = P['test']
    pred = {}

    # 1-2. Líneas base
    pred['Persistencia'] = gate.copy()
    p, coef = ajustar_arima(Xtr)
    pred[f'ARIMA({p},1,0)'] = predecir_arima(Xte, p, coef, H)

    # 3-4. GRU y LSTM solo con datos propios
    for tipo in ('gru', 'lstm'):
        m = construir_rnn(tipo)
        entrenar(m, Xtr, ytr, Xva, yva, nombre=f'{tipo.upper()} propio')
        pred[f'{tipo.upper()} propio'] = predecir_glucosa(m, Xte, gate); modelos[(H, f'{tipo.upper()} propio')] = m

    # Datos de Ohio
    Xo, yo, *_ = juntar(ohio_train, H)
    Xot, yot, *_ = juntar(ohio_test, H)

    # 5. LSTM con dataset unificado (Ohio train + propio train en un solo dataset)
    Xu = np.concatenate([Xo, Xtr]); yu = np.concatenate([yo, ytr])
    w_prop = (len(Xo) / len(Xtr)) if PESO_PROPIO_UNIFICADO == 'balanceado' else float(PESO_PROPIO_UNIFICADO)
    wu = np.concatenate([np.ones(len(Xo)), np.full(len(Xtr), w_prop)]).astype('float32')
    wu = wu / wu.mean()
    perm = np.random.default_rng(SEMILLA).permutation(len(Xu))
    m = construir_rnn('lstm')
    entrenar(m, Xu[perm], yu[perm], Xva, yva, pesos=wu[perm], batch=256, nombre='LSTM dataset unificado')
    pred['LSTM dataset unificado'] = predecir_glucosa(m, Xte, gate); modelos[(H, 'LSTM dataset unificado')] = m

    # 6. LSTM transfer learning: pre-entrenamiento Ohio → fine-tuning propio
    m = construir_rnn('lstm')
    entrenar(m, Xo, yo, Xot, yot, batch=256, nombre='LSTM pre-entrenado Ohio')
    pred['LSTM pre-entrenado Ohio (sin ajuste)'] = predecir_glucosa(m, Xte, gate)
    entrenar(m, Xtr, ytr, Xva, yva, lr=LR_FINETUNE, nombre='LSTM transfer (fine-tuning)')
    pred['LSTM transfer learning'] = predecir_glucosa(m, Xte, gate); modelos[(H, 'LSTM transfer learning')] = m

    for nombre, pr in pred.items():
        resultados.append({'Horizonte_min': H * 5, 'Modelo': nombre, **metricas(gfte, pr, tte)})
    predicciones[H] = {'real': gfte, 'tiempo': tte, 'X': Xte, 'g_act': gate, **pred}

tabla = pd.DataFrame(resultados)
tabla.to_csv(os.path.join(CARPETA_RES, 'tabla_resultados.csv'), index=False)
pd.set_option('display.width', 200)
tabla.round(2)

## 12. Comparativa gráfica

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONTES), figsize=(7 * len(HORIZONTES), 5), squeeze=False)
for ax, H in zip(axes[0], HORIZONTES):
    t = tabla[tabla['Horizonte_min'] == H * 5].sort_values('RMSE')
    lo = t['RMSE_IC95'].str.strip('[]').str.split(',').str[0].astype(float)
    hi = t['RMSE_IC95'].str.strip('[]').str.split(',').str[1].astype(float)
    colores = ['#1a73e8' if 'LSTM' in m or 'GRU' in m else '#9e9e9e' for m in t['Modelo']]
    ax.barh(t['Modelo'], t['RMSE'], xerr=[t['RMSE'] - lo, hi - t['RMSE']], color=colores, capsize=3)
    ax.invert_yaxis(); ax.set_xlabel('RMSE (mg/dL) con IC 95 %'); ax.set_title(f'Horizonte {H*5} min')
plt.tight_layout(); plt.savefig(os.path.join(CARPETA_RES, 'comparativa_rmse.png'), dpi=150); plt.show()

In [ ]:
def grilla_clarke(ax, ref, pred, titulo):
    ax.scatter(ref, pred, s=2, alpha=0.25, color='#1a73e8')
    ax.plot([0, 400], [0, 400], 'k:', lw=0.8)
    for xs, ys in [([0, 175/3], [70, 70]), ([175/3, 400/1.2], [70, 400]), ([70, 70], [84, 400]), ([0, 70], [180, 180]),
                   ([70, 290], [180, 400]), ([70, 70], [0, 56]), ([70, 400], [56, 320]), ([180, 180], [0, 70]),
                   ([180, 400], [70, 70]), ([240, 240], [70, 180]), ([240, 400], [180, 180]), ([130, 180], [0, 70])]:
        ax.plot(xs, ys, 'k-', lw=0.8)
    for txt, x, y in [('A', 30, 15), ('B', 370, 260), ('B', 280, 370), ('C', 160, 370), ('C', 160, 15),
                      ('D', 30, 140), ('D', 370, 120), ('E', 30, 370), ('E', 370, 15)]:
        ax.text(x, y, txt, fontsize=13, weight='bold')
    ax.set_xlim(0, 400); ax.set_ylim(0, 400); ax.set_xlabel('Glucosa real (mg/dL)'); ax.set_ylabel('Predicción (mg/dL)')
    ax.set_title(titulo)

for H in HORIZONTES:
    t = tabla[(tabla['Horizonte_min'] == H * 5) & tabla['Modelo'].str.contains('LSTM|GRU')].sort_values('RMSE')
    mejor = t['Modelo'].iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    grilla_clarke(axes[0], predicciones[H]['real'], predicciones[H]['Persistencia'], f'Persistencia — {H*5} min')
    grilla_clarke(axes[1], predicciones[H]['real'], predicciones[H][mejor], f'{mejor} — {H*5} min')
    plt.tight_layout(); plt.savefig(os.path.join(CARPETA_RES, f'clarke_{H*5}min.png'), dpi=150); plt.show()

## 13. Incertidumbre: Monte Carlo Dropout
Se repite la predicción del mejor modelo recurrente con el *dropout* activo (las capas de BatchNormalization quedan en modo inferencia). La dispersión de las pasadas da una banda del 95 %. Se informa la **cobertura** (qué % de los valores reales cae dentro de la banda; idealmente ~95 %) y el **ancho medio**.

In [ ]:
def mc_dropout(m, X, g_act, T=MC_MUESTRAS, lote=4096):
    salidas = []
    for _ in range(T):
        partes = []
        for i in range(0, len(X), lote):
            x = tf.convert_to_tensor(X[i:i + lote])
            for capa in m.layers:
                x = capa(x, training=isinstance(capa, Dropout))
            partes.append(x.numpy().ravel())
        salidas.append(g_act + np.concatenate(partes) * ESCALA_DELTA)
    s = np.stack(salidas)
    return s.mean(0), np.percentile(s, 2.5, 0), np.percentile(s, 97.5, 0)

incert = []
for H in HORIZONTES:
    t = tabla[(tabla['Horizonte_min'] == H * 5) & tabla['Modelo'].isin([k[1] for k in modelos if k[0] == H])].sort_values('RMSE')
    mejor = t['Modelo'].iloc[0]
    d = predicciones[H]
    media, lo, hi = mc_dropout(modelos[(H, mejor)], d['X'], d['g_act'])
    cobertura = np.mean((d['real'] >= lo) & (d['real'] <= hi)) * 100
    incert.append({'Horizonte_min': H * 5, 'Modelo': mejor, 'Cobertura_95_%': cobertura, 'Ancho_medio_mg/dL': np.mean(hi - lo)})
    d['mc'] = (media, lo, hi, mejor)

    n = min(576, len(d['real']))   # últimas 48 h del test
    tt = d['tiempo'][-n:]
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.fill_between(tt, lo[-n:], hi[-n:], color='#e53935', alpha=0.15, label='Banda 95 % (MC Dropout)')
    ax.plot(tt, d['real'][-n:], color='#1a73e8', lw=1.2, label='Glucosa real')
    ax.plot(tt, d[mejor][-n:], color='#e53935', lw=1, ls='--', label=f'{mejor}')
    ax.plot(tt, d['Persistencia'][-n:], color='#9e9e9e', lw=0.8, ls=':', label='Persistencia')
    ax.axhspan(70, 180, color='green', alpha=0.05)
    ax.set_ylabel('mg/dL'); ax.set_title(f'Predicción a {H*5} min — últimas 48 h del test (cobertura {cobertura:.0f} %)')
    ax.legend(loc='upper right'); plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_RES, f'prediccion_incertidumbre_{H*5}min.png'), dpi=150); plt.show()

tabla_incert = pd.DataFrame(incert)
tabla_incert.to_csv(os.path.join(CARPETA_RES, 'tabla_incertidumbre.csv'), index=False)
tabla_incert.round(1)

## 14. Guardar modelos y resumen

In [ ]:
for (H, nombre), m in modelos.items():
    m.save(os.path.join(CARPETA_RES, f"modelo_{nombre.replace(' ', '_').lower()}_{H*5}min.keras"))

resumen = {
    'fecha': datetime.now().isoformat(timespec='minutes'),
    'datos_propios': {'desde': str(df_propio.index.min()), 'hasta': str(df_propio.index.max()),
                      'lecturas': int(df_propio['glucosa'].notna().sum())},
    'ohio_pacientes': len(ohio_train), 'features': FEATURES, 'ventana_min': VENTANA * 5,
    'horizontes_min': [h * 5 for h in HORIZONTES], 'split': SPLIT,
    'peso_propio_unificado': PESO_PROPIO_UNIFICADO,
}
with open(os.path.join(CARPETA_RES, 'resumen_experimento.json'), 'w') as f:
    json.dump(resumen, f, indent=2, default=str)

print('Archivos en', CARPETA_RES)
for f in sorted(os.listdir(CARPETA_RES)):
    print('  ', f)
print('\nMejor modelo por horizonte (RMSE):')
print(tabla.loc[tabla.groupby('Horizonte_min')['RMSE'].idxmin(), ['Horizonte_min', 'Modelo', 'RMSE', 'RMSE_IC95', 'Clarke_A_%']].round(2).to_string(index=False))

## 15. Cómo leer los resultados (para el capítulo de la Entrega 2)

- **Persistencia** es el piso. Si una red no la supera claramente a 30 min, no aporta valor clínico.
- **Superposición de intervalos**: si los IC 95 % del RMSE de dos modelos se solapan mucho, la diferencia **no es concluyente** con estos datos. Decirlo así en el TFM es más sólido que afirmar que uno "gana".
- **Unificado vs transfer learning** responde la pregunta del profesor:
  - el *dataset unificado* aprende de todos a la vez y usa el peso de las muestras para que los datos propios no queden diluidos;
  - el *transfer learning* aprende primero patrones generales y después se especializa.
  - La fila **"pre-entrenado Ohio (sin ajuste)"** muestra cuánto rinde un modelo poblacional **sin** datos del paciente: es un indicador de *domain shift* entre sensores (Enlite vs Guardian 4) y entre terapias (bomba de 2018 vs SmartGuard 780G).
- **Clarke A+B ≥ 99 %** es lo habitual en la literatura para considerar una predicción clínicamente aceptable.
- **Incertidumbre**: si la cobertura queda muy por debajo del 95 %, la banda de MC Dropout es *optimista*. Es una limitación conocida del método y conviene mencionarla.
- **Limitación a declarar**: la 780G administra micro-bolos automáticos (SmartGuard) que no están en Nightscout como bolos. El modelo solo ve los bolos manuales y de corrección, así que el IOB es una subestimación.